# QUERY PORFILE

Diagnostic tool that gives you a detailed insights into the execution of a query. It contains a visual representation of the query plan, with all nodes and links represented. Execution details and statistics are provided for each node as well as the overall query.

When to use?
Whenever diagnostic information about a query is needed.

# Query Insights

feature that builds on the Query Profie by automatically detecting conditions that might affect performance and offering actionable recommendations.

It is account_usage view and can be used same way as other usage view. This view has specific columns that share information about queries, warehouse and three key columns
- insight_type_id
- message
- suggestion

We can use query_id to check if there are any insights available in the table and get recommendations to improve performance

example

select query_id, insight_insatnce_id, insight_type_id, message, suggestions
from Snowflake.account_usage.quer_insights
where query_id = '<query_id>';

# Bytes Spilled to Storage

It refers to 'Temporary data' that a virtual warehouse writes to disk (local SSD or remote cloud storage) when an operation runs out of available memory.

Impact : This spillage degrades Query Performance because Disk I/O is much slower than memory access.

When does Spilling Happend

Disk spilling is almost always about resource pressure: the data involved in operations like 'ORDER BY','GROUP BY' or joins simple doesn't fit in memory

Major cause
- Large Data operations : sorting, grouping or joining massive datasets
- Warehouse size : using a small or overload virtual warehouse
- COncurrency : many queries ran at once, competing for memory
- Suboptimal SQL : complex or inefficient query designs

<img src="Image3_spillage.png"  />

# Inefficient Pruning

This occurs when 
- Queries scan too many Micro- partitions, often due to poor clustering. improper data types in filter
(using to_char on dates) or using complex expressions in Where.
- resulting in high I/O, increased query time and high cost

Common Cause and solution
1. Weak Clustering: If table data is not sorted or is only partially sorted, snowflake cannot efficeintly skip irrelevant partitions.
Sol : Implement automatic clustering on frequently filtered columns

2. Unsupported Filter expressions: Functions applied directly to columns prevent optimization. e.g: year(column)>2024
Sol: Use direct column comparisons such as where column >= '2024-01-01' and column < '2025-01-01'

3. Subquery Predicates: Queries using where x in select may not always prune effectively.
Sol : Use join claues or CTEs to allow for better partition eliination

4. Unsupported Data Types: Using specific data types can sometimes limit metadata usage

5. Frequent small updates: Frequent updates and deletes fragment data, making pruing less effective

# Exploding Joins

It occurs when a join operation produces an unexpectedly and excessivley large number of rows, far exceeding the input data size.
It leads to severe performance degradation, long running queries, memory spillage, increased comupte costs and potentially query failures

Common cause
- missing or incomplete join condtion
- duplicate values in join keys
- non equi joins
- complex querries/flipped joins

# Queueing

It occurs when a virtual warehouse has insufficient compute resources to handle the volume of incoming queries, causing them to wait in FIFO queue until capcity frees up.
- reason : high concurrency or resource exhaustion
Sol:

- multiple cluster warehouse : enable auto_scaling to add compute clusters automatically
- resize warehouse: increase warehouse size to provide more resources per query
- Query Optimization : review query history to optimize poorly performing sql

# ACCOUNT USAGE

QUERY ATTRIBUTION HISTORY   :
this tracks query specific cost and allows users to accurately measure and analyze the resource consumption of their queues, enabling more effective cost management and optimization

Use: vailuable tool for gaining insights into the cost of individual queries executed within your account. It provides detailed information about resource consumption of each query, enabling you to optimize perforamnce and manage cost efectively.

Note : Latency : upto 6 hrs
Role : usage_viewer database role
short running queries : queries less than 100 ms running time would not be included in this view.

# Query History

| feature | Information schema | account usage | snowsight ui |
|--- |---| ---| ---|
| retention | 7 days | 365 days | 14 days |
|Latency | Real time | 45 mins - 3 hrs | real time |
| best for | troubleshooting current issues | long term auditing/billing | visualizing query plans |

# Grouping similar workloads

1. Functional grouping : 

- ETL/ELT : group all COPY INTO or INSERT tasks here. These are usually credit heavy and can run on smaller warehouse for a longer time
- BI : required low latency and high concurrency
- Transformation (dbt/task): group scheduled sql transformations. This allows you to track exactly how much it costs to build your data models
- Adhoc: group your analyst and data scientist. Queries are unpredictable and often need large warehouse for heavy joins

2. Technical Grouping:
- Horizantal scaling: (multi cluster) : group workload with high concurrency
standard scaling: add clusters when queue forms. best for BI
economic scaling: only adds clusters if there is enough work to keep busy for 6 mins
- Vertical scaling (warehouse size) : group workloads that needs to process massive volumes of data( large scans/joins) ona large warehouse. This provides more memory and local SSD space to prevent spilling to remote storage

# Query Acceleration Service (QAS)
feature that improves the performance of large, scan intensive queries by offloading portions of the processing work to seperate, shared compute resources. This allows the primary warehouse to focus on other tasks and can significantly reduce the execution time of eligible queries without requiring a full warehouse resize.

How it works

- automatic detection : snowflake's optimizer analyzes the query plan at runtime to suggest query acceleration serivce for portions of the query
- Offloading work : If elligible, portions of these work are offloaded to dedicated QAS serverless compute resources, that process the data in parallel.
- pay per use : serivce is billed seperately from the main warehouse on per second basis, only consuming credits when used
- no code changes: it integrates seamlessly with existing setups, requiring no changes to SQL code or pipeline redesign

Enabling & Config

requires Enterprise, ebnabled at warehouse level using

CREATE WAREHOUSE MY_WH WITH ENABLE_QUERY_ACCELERATION = TRUE;
ALTER WAREHOUSE MY_WH SET ENABLE_QUERY_ACCELERATION = TRUE;

Control cos with scale factor: The query_acceleration_max_scale_factor limits the max amt of additional compute reources the service can lease. default is 8 and max is 16 ( 2,4,8,16)

ALTER WAREHOUSE MY_WH SET QUERY_ACCELERATION_MAX_SCALE_FACTOR = 4;

# SEARCH OPTIMIZATION SERVICE (SAS)

problem: you have terabytes of data and you need to do a point lookup, like select * from table wher id='r123';

designed to significantly impprove the performance of specific, highly selective queries , often referred to as point lookup on large tables;

syntax

ALTER TABLE MY_TABLE ADD SEARCH OPTIMIZATION;

HOW IT WORKS:
- Creates and maintains a persistent hidden data structure called <font color= red>search access path</font>.
- path keeps track which data values reside in which micro partitons
- during query, the optimizer uses this search access path to prune unneccesary micro partition , thus reducing amt of data read and speeds the query

FEATURES

- Improve query performance
- Automatic maintainance
- Broad data type support
- substring and text search

When to use

- queries run for several mins
- highly selective filters that return small rows
- cols with high no of distinct values
- large tables that are not clustered or are frequently queried on columns other than the clustering key

| Feature | Goal | query type | User effort |
| --- | --- | --- | --- |
| SAS | find specific rows faster | where id = 123 (point lookups) | low |
| QAS | scale compute for huge scans | select * from large table or large filters | medium (set scale factor) |
| Clustering keys | Organize data for pruning | where data between | high (design ) |


